In [10]:
import numpy as np
import json
import os
from pathlib import Path

import torch
from transformers import AutoProcessor, WhisperForConditionalGeneration, AutoModelForSpeechSeq2Seq, pipeline
import pandas as pd
from tqdm import tqdm

import warnings
warnings.filterwarnings(action='ignore')

In [2]:
DS_PATH = "../data/golos_opus/test_opus/crowd"
CROWD_JSONL_PATH = os.path.join(DS_PATH, "crowd.jsonl")
OUTPUT_DIR = "./transcriptions_hf"

MODEL_NAME = "lorenzoncina/whisper-small-ru"
LANGUAGE = "russian"
TASK = "transcribe"

DEVICE = "cuda" if torch.cuda.is_available() else 'cpu'
BATCH_SIZE=8
print("Device: ", DEVICE)

Device:  cuda


In [3]:
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

In [4]:
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    trust_remote_code=True,
)

model.to(DEVICE)
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/840 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

In [12]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=DEVICE,
    dtype=torch.float16,
)


In [5]:
audio_files = []
with open(CROWD_JSONL_PATH, 'r') as file:
    for line in file:
        if line.strip():
            entry = json.loads(line)
            audio_filename = entry.get('audio_filepath', None)
            if audio_filename:
                audio_path = os.path.join(DS_PATH, audio_filename)
                audio_files.append({
                    'original_data': entry,
                    'audio_path': audio_path
                })

print(f"Loaded {len(audio_files)} from JSONL")

Loaded 9994 from JSONL


In [6]:
valid_entries = []
for entry in audio_files:
    if os.path.exists(entry['audio_path']):
        valid_entries.append(entry)
    else:
        print(f"Not found: {entry['audio_path']}")

print(f"For inference: {len(valid_entries)} files.")

For inference: 9994 files.


In [7]:
def transcribe_audio(batch_paths):
    try:
        result = pipe(
            batch_paths,
            batch_size=len(batch_paths),
            generate_kwargs={"task": TASK, "language": LANGUAGE}
        )
        return {
            "status": True,
            "result": result
        }
    except Exception as e:
        print("Exception occured: ", e)
        return {
            "status": False,
            "exception": e
        }

In [13]:
results = []
for i in tqdm(range(0, 106, BATCH_SIZE), desc="Transcribation"):
    batch_entries = valid_entries[i:i+BATCH_SIZE]
    batch_paths = [file["audio_path"] for file in batch_entries]

    output = transcribe_audio(batch_paths)
    for entry in batch_entries:
        if "exception" in output:
            result = {
                'audio_path': entry['audio_path'],
                'original_data': entry['original_data'],
                'transcription': {'success': False, 'error': output['exception']}
            }
        else:
            result = {
                'audio_path': entry['audio_path'],
                'original_data': entry['original_data'],
                'transcription': {
                    'success': True,
                    'text': output['result']
                }
            }
        results.append(result)

Transcribation:   0%|                                                                                                      | 0/14 [00:00<?, ?it/s][transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please 

In [19]:
results

[{'audio_path': '../data/golos_opus/test_opus/crowd/files/e632f7d39c15e7edfc665b91e6f2071f.opus',
  'original_data': {'id': 'e632f7d39c15e7edfc665b91e6f2071f',
   'audio_filepath': 'files/e632f7d39c15e7edfc665b91e6f2071f.opus',
   'text': 'афина воспроизведи музыку вперемешку',
   'duration': 4.9},
  'transcription': {'success': True,
   'text': [{'text': 'Афина, воспроизведи музыку в перемешку.'},
    {'text': 'Найти сериал Григорий эр.'},
    {'text': 'Прямой эфир опыл на чистую и на этот Тоттенхен.'},
    {'text': 'Ильвиром Ивановичем Ворончихиным.'},
    {'text': 'Можешь показать киношку «Исходный кот».'},
    {'text': 'Изменился ли курс рубля к американскому доллару со вчерашнего дня?'},
    {'text': 'Найчи фильм плохей, парни, первую часть.'},
    {'text': 'Эпизод тридцать один, сезона тринадцать, «Фильма люди».'}]}},
 {'audio_path': '../data/golos_opus/test_opus/crowd/files/5db5df8bb9e3b6660b2a04b34d4a355d.opus',
  'original_data': {'id': '5db5df8bb9e3b6660b2a04b34d4a355d',
   '

In [26]:
sample_batch = results[0]['transcription']['text'] + results[8]['transcription']['text']
ground_truths = [results[i]['original_data']['text'] for i in range(16)]
for pred, gd in zip(sample_batch, ground_truths):
    print("Predicted: ", pred['text'], "| Actual: ", gd)

Predicted:  Афина, воспроизведи музыку в перемешку. | Actual:  афина воспроизведи музыку вперемешку
Predicted:  Найти сериал Григорий эр. | Actual:  найти сериал григорий р
Predicted:  Прямой эфир опыл на чистую и на этот Тоттенхен. | Actual:  прямой эфир апл манчестер юнайтед тоттенхэм
Predicted:  Ильвиром Ивановичем Ворончихиным. | Actual:  ильвиром ивановичем ворончихиным
Predicted:  Можешь показать киношку «Исходный кот». | Actual:  можешь показать киношку исходный код
Predicted:  Изменился ли курс рубля к американскому доллару со вчерашнего дня? | Actual:  изменился ли курс рубля к американскому доллару со вчерашнего дня
Predicted:  Найчи фильм плохей, парни, первую часть. | Actual:  найти фильм плохие парни первую часть
Predicted:  Эпизод тридцать один, сезона тринадцать, «Фильма люди». | Actual:  эпизод тридцать один сезона тринадцать фильма люди
Predicted:  Купить моющее средство! | Actual:  купить моющее средство
Predicted:  – Ноль… – Триста восемь, триста восемьнадцать, трист

In [27]:
import re
import jiwer

In [28]:
def normalize_text(text_str):
    text_lower = text_str.lower()
    text_normalized = re.sub(r'^[\w\s]', '', text_lower)
    text_stripped = re.sub(r'\s+', ' ', text_normalized).strip()
    return text_stripped

In [29]:
preds_norm = [normalize_text(t['text']) for t in sample_batch]
ground_norm = [normalize_text(gd) for gd in ground_truths]

In [30]:
for i, (pred, ref) in enumerate(zip(preds_norm, ground_norm)):
    cer = jiwer.cer(ref, pred)
    wer = jiwer.wer(ref, pred)
    print(f"Index {i} | CER: {cer:.4f} | WER: {wer:.4f}")

avg_wer = jiwer.wer(ground_norm, preds_norm)
avg_cer = jiwer.cer(ground_norm, preds_norm)

print("Average CER: %.4f" % avg_cer)
print("Average WER: %.4f" % avg_wer)

Index 0 | CER: 0.0857 | WER: 0.7500
Index 1 | CER: 0.0909 | WER: 0.2500
Index 2 | CER: 0.3810 | WER: 1.1667
Index 3 | CER: 0.0323 | WER: 0.3333
Index 4 | CER: 0.1143 | WER: 0.4000
Index 5 | CER: 0.0156 | WER: 0.1000
Index 6 | CER: 0.1667 | WER: 0.6667
Index 7 | CER: 0.1020 | WER: 0.5714
Index 8 | CER: 0.0476 | WER: 0.3333
Index 9 | CER: 0.1449 | WER: 0.6667
Index 10 | CER: 0.1489 | WER: 0.4286
Index 11 | CER: 0.5312 | WER: 0.8333
Index 12 | CER: 0.0270 | WER: 0.2000
Index 13 | CER: 0.3214 | WER: 0.5000
Index 14 | CER: 0.1136 | WER: 0.4286
Index 15 | CER: 0.1034 | WER: 0.4000
Average CER: 0.1446
Average WER: 0.5000


In [31]:
with open('gd.jsonl', 'w') as f:
    for entry in ground_norm:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

In [33]:
with open('whisper-small-ru' + ".jsonl", 'w') as f:
    for entry in preds_norm:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')